# DIGITAL TRANSACTION RISK ANALYSIS
## Unit III: Inferential Statistics and Hypothesis Testing

This unit performs exactly 2 hypothesis tests as required by the project guidelines:
1. **Two-sample t-test**: Compare mean transaction amounts between fraudulent vs legitimate transactions
2. **Chi-square test**: Test independence between transaction category and fraud occurrence

For each test, we state hypotheses, calculate statistics, and make decisions using alpha = 0.05.

In [ ]:
# ================================================================
# SETUP: DEPENDENCIES AND DATA LOADING
# ================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')

# Configure display settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Load the dataset
try:
    print("Loading dataset for inferential analysis...")
    df = pd.read_csv('fraudTrain.csv')
    print(f"✓ Dataset loaded successfully: {df.shape[0]:,} rows x {df.shape[1]} columns")
except FileNotFoundError:
    print("❌ Error: fraudTrain.csv not found in current directory.")

Dataset loaded successfully.


# Unit III: Inferential Statistics and Hypothesis Testing

This section performs exactly 2 hypothesis tests as required by the project guidelines.

In [ ]:
print('\n' + '=' * 70)
print('TEST 1: TWO-SAMPLE T-TEST (Transaction Amount vs Fraud Status)')
print('=' * 70)

# Separate amounts by fraud status
fraud_amounts = df[df['is_fraud'] == 1]['amt']
legit_amounts = df[df['is_fraud'] == 0]['amt']

print(f'\nSample Sizes:')
print(f'  • Fraudulent transactions: {len(fraud_amounts):,}')
print(f'  • Legitimate transactions: {len(legit_amounts):,}')

print(f'\nDescriptive Statistics:')
print(f"{'':20} {'Mean':>12} {'Std Dev':>12} {'Median':>12}")
print('-' * 50)
print(f"{'Fraudulent':<20} ${fraud_amounts.mean():>11.2f} ${fraud_amounts.std():>11.2f} ${fraud_amounts.median():>11.2f}")
print(f"{'Legitimate':<20} ${legit_amounts.mean():>11.2f} ${legit_amounts.std():>11.2f} ${legit_amounts.median():>11.2f}")

print(f'\nHypothesis:')
print(f'  H₀ (Null):       Mean(Fraud) = Mean(Legitimate)')
print(f'  H₁ (Alternative): Mean(Fraud) ≠ Mean(Legitimate)')
print(f'  Significance Level: α = 0.05')

# Test for equal variances
levene_stat, p_levene = stats.levene(fraud_amounts, legit_amounts)
print(f'\nLevene\'s Test for Equal Variances:')
print(f'  Statistic: {levene_stat:.4f}, p-value: {p_levene:.4e}')
if p_levene < 0.05:
    print(f'  → Variances are NOT equal (use Welch\'s t-test)')
else:
    print(f'  → Variances are equal')

# Perform Welch's t-test
t_stat, p_val = stats.ttest_ind(fraud_amounts, legit_amounts, equal_var=False)

print(f'\nWelch\'s t-Test Results:')
print(f'  t-statistic: {t_stat:.4f}')
print(f'  p-value:     {p_val:.4e}')

# Calculate effect size (Cohen's d)
diff = fraud_amounts.mean() - legit_amounts.mean()
pooled_std = np.sqrt((fraud_amounts.std()**2 + legit_amounts.std()**2) / 2)
cohens_d = diff / pooled_std
print(f'  Cohen\'s d (effect size): {cohens_d:.4f}')

print(f'\nDecision (α = 0.05):')
if p_val < 0.05:
    print(f'  ✓ REJECT H₀: Fraudulent and legitimate transactions have SIGNIFICANTLY')
    print(f'    different mean amounts.')
else:
    print(f'  ✗ FAIL TO REJECT H₀: No significant difference detected.')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist([legit_amounts, fraud_amounts], bins=50, label=['Legitimate', 'Fraudulent'], 
             color=['steelblue', 'coral'], alpha=0.7, edgecolor='black')
axes[0].set_title('Distribution of Transaction Amounts by Fraud Status', fontweight='bold')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

axes[1].boxplot([legit_amounts, fraud_amounts], labels=['Legitimate', 'Fraudulent'])
axes[1].set_title('Box Plot: Amount by Fraud Status', fontweight='bold')
axes[1].set_ylabel('Amount ($)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


HYPOTHESIS TEST 1: TWO-SAMPLE T-TEST (AMOUNT VS FRAUD)
Null Hypothesis (H0): Mean(Fraud) = Mean(Legit)
Alt Hypothesis (H1) : Mean(Fraud) != Mean(Legit)

Levene's Test for Equal Variances: p-value = 0.0000e+00

Welch's t-test Results:
  t-statistic : 102.8047
  p-value     : 0.0000e+00
  Effect Size (Cohen's d): 1.5618

Decision: Reject H0 at alpha=0.05. There is a significant difference in means.


In [ ]:
print('\n' + '=' * 70)
print('TEST 2: CHI-SQUARE TEST (Transaction Category vs Fraud Independence)')
print('=' * 70)

# Create contingency table
contingency_table = pd.crosstab(df['category'], df['is_fraud'])
print(f'\nContingency Table (Category vs Fraud Status):')
print(contingency_table)

# Perform chi-square test
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency_table)

print(f'\nHypothesis:')
print(f'  H₀ (Null):       Transaction category and fraud are INDEPENDENT')
print(f'  H₁ (Alternative): There is an ASSOCIATION between category and fraud')
print(f'  Significance Level: α = 0.05')

print(f'\nChi-Square Test Results:')
print(f'  Chi-square statistic: {chi2:.4f}')
print(f'  p-value:              {p_chi:.4e}')
print(f'  Degrees of freedom:   {dof}')

# Calculate effect size (Cramér's V)
n = contingency_table.sum().sum()
min_dim = min(contingency_table.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
print(f'  Cramér\'s V (effect size): {cramers_v:.4f}')

print(f'\nDecision (α = 0.05):')
if p_chi < 0.05:
    print(f'  ✓ REJECT H₀: Fraud occurrence is SIGNIFICANTLY ASSOCIATED with')
    print(f'    transaction category.')
else:
    print(f'  ✗ FAIL TO REJECT H₀: No significant association detected.')

# Visualization: Stacked bar chart
fig, ax = plt.subplots(figsize=(12, 6))
contingency_pct = contingency_table.div(contingency_table.sum(axis=1), axis=0) * 100

contingency_pct.plot(kind='bar', stacked=False, ax=ax, color=['steelblue', 'coral'], 
                     edgecolor='black', alpha=0.8)
ax.set_title('Fraud Rate by Transaction Category', fontweight='bold', fontsize=12)
ax.set_xlabel('Transaction Category')
ax.set_ylabel('Percentage (%)')
ax.legend(title='Fraud Status', labels=['Legitimate', 'Fraudulent'])
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()


HYPOTHESIS TEST 2: CHI-SQUARE TEST OF INDEPENDENCE
Null Hypothesis (H0): Transaction category and fraud are independent.
Alt Hypothesis (H1) : There is an association between category and fraud.

Chi-Square Test Results:
  Chi2 stat   : 6486.0033
  p-value     : 0.0000e+00
  Effect Size (Cramer's V): 0.0707

Decision: Reject H0. Fraud occurrence depends on the transaction category.


In [4]:
print('\n' + '='*60)
print('UNIT III COMPLETE')
print('='*60)


UNIT III COMPLETE
